In [ ]:
import os
import sys
import json
import re
import numpy as np
from difflib import SequenceMatcher
from openai import OpenAI

# Add parent directory to path for imports
sys.path.insert(0, os.path.abspath('..'))

# -------------------------------------------------------------------
# CONFIGURATION
# -------------------------------------------------------------------

DATA_DIR = "../data"
TRANSLATIONS_DIR = os.path.join(DATA_DIR, "translations")
EMBEDDINGS_DIR = os.path.join(DATA_DIR, "embeddings")

# Language-specific dictionary paths (using German as reference)
QURAN_DICT_PATH = os.path.join(TRANSLATIONS_DIR, "quran", "de.json")
ATHAN_DICT_PATH = os.path.join(TRANSLATIONS_DIR, "athan", "de.json")
QURAN_EMBEDDINGS_PATH = os.path.join(EMBEDDINGS_DIR, "quran_embeddings.json")

TRANSLATION_MODEL = "gpt-4o-mini"
EMBEDDING_MODEL = "text-embedding-3-large"

# RAG settings
RAG_MIN_SIMILARITY = 0.60
RAG_TOP_K = 5
ATHAN_MATCH_THRESHOLD = 0.75

# -------------------------------------------------------------------
# LOAD DATA
# -------------------------------------------------------------------

def load_json(path):
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    return {}

quran_dict = load_json(QURAN_DICT_PATH)
athan_dict = load_json(ATHAN_DICT_PATH)
quran_embeddings = load_json(QURAN_EMBEDDINGS_PATH)

print(f"Quran dictionary: {len(quran_dict)} verses")
print(f"Athan dictionary: {len(athan_dict)} phrases")
print(f"Quran embeddings: {len(quran_embeddings)} vectors")

if not quran_embeddings:
    print("\n⚠️  WARNING: quran_embeddings.json not found or empty!")
    print("   RAG matching will not work. Download from:")
    print("   https://drive.google.com/file/d/1qXJwjzoWM-YpN5O_YYjapg6Sf7uC1CUQ/view")

# OpenAI client
client = OpenAI()

In [ ]:
# -------------------------------------------------------------------
# HELPER FUNCTIONS
# -------------------------------------------------------------------

def normalize_arabic(text: str) -> str:
    """Normalize Arabic text for matching (remove diacritics, normalize letters)."""
    text = text.strip()
    # Remove harakat (diacritical marks)
    harakat = re.compile(r'[\u0610-\u061A\u064B-\u065F\u0670\u06D6-\u06ED]')
    text = harakat.sub('', text)
    # Normalize Alef variants
    text = text.replace("أ", "ا").replace("إ", "ا").replace("آ", "ا")
    # Normalize taa marbuta
    text = text.replace("ة", "ه")
    # Normalize spaces
    text = re.sub(r'\s+', ' ', text)
    return text


def fuzzy_match_athan(text: str) -> tuple[float, str | None, str | None]:
    """Match text against Athan dictionary using fuzzy matching."""
    text_norm = normalize_arabic(text)
    best_score = 0.0
    best_de = None
    best_ar = None

    for ar, de in athan_dict.items():
        ar_norm = normalize_arabic(ar)
        score = SequenceMatcher(None, ar_norm, text_norm).ratio()
        if score > best_score:
            best_score = score
            best_de = de
            best_ar = ar

    return best_score, best_de, best_ar


def get_text_embedding(text: str) -> np.ndarray:
    """Get embedding for text from OpenAI."""
    text = (text or "").strip()
    if not text:
        return np.zeros(1, dtype=np.float32)

    try:
        resp = client.embeddings.create(
            model=EMBEDDING_MODEL,
            input=[text]
        )
        return np.array(resp.data[0].embedding, dtype=np.float32)
    except Exception as e:
        print(f"Error getting embedding: {e}")
        return np.zeros(1, dtype=np.float32)


def get_quran_embedding(arabic_verse: str) -> np.ndarray:
    """Get cached embedding for a Quran verse."""
    cached = quran_embeddings.get(arabic_verse)
    if cached is not None:
        return np.array(cached, dtype=np.float32)
    return np.zeros(1, dtype=np.float32)


def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    """Compute cosine similarity between two vectors."""
    if a.size <= 1 or b.size <= 1:
        return 0.0
    denom = (np.linalg.norm(a) * np.linalg.norm(b)) + 1e-8
    return float(np.dot(a, b) / denom)

In [ ]:
# -------------------------------------------------------------------
# RAG MATCHING
# -------------------------------------------------------------------

def match_quran_rag(text: str, min_similarity: float = RAG_MIN_SIMILARITY, top_k: int = RAG_TOP_K):
    """
    RAG matching for Quran verses:
    - Embed the input text
    - Compare with all Quran verse embeddings via cosine similarity
    - Return top-k matches above the threshold
    
    Returns:
        List of (score, arabic_verse, german_translation)
    """
    txt = (text or "").strip()
    if not txt or not quran_dict or not quran_embeddings:
        return []

    query_emb = get_text_embedding(txt)
    if query_emb.size <= 1:
        return []

    matches = []
    for ar, de in quran_dict.items():
        verse_emb = get_quran_embedding(ar)
        if verse_emb.size <= 1:
            continue
        score = cosine_similarity(query_emb, verse_emb)
        if score >= min_similarity:
            matches.append((score, ar, de))

    # Sort by score descending and limit to top_k
    matches.sort(key=lambda x: x[0], reverse=True)
    return matches[:top_k]

In [ ]:
# -------------------------------------------------------------------
# TRANSLATION
# -------------------------------------------------------------------

def translate_text(text: str, context: str = "") -> str:
    """
    Full translation pipeline:
    1. Check for Athan phrases (direct dictionary match)
    2. Find Quran verse candidates via RAG
    3. Translate with GPT, using RAG hints for accuracy
    """
    txt = (text or "").strip()
    if not txt:
        return ""

    # 1) Athan detection
    score_athan, de_athan, ar_athan = fuzzy_match_athan(txt)
    if score_athan >= ATHAN_MATCH_THRESHOLD:
        print(f"✓ Athan detected: '{ar_athan}' (score={score_athan:.2f})")
        return de_athan

    # 2) Quran RAG matching
    quran_hint = ""
    quran_matches = match_quran_rag(txt)

    if quran_matches:
        print(f"✓ Found {len(quran_matches)} Quran candidate(s)")
        blocks = []
        for idx, (score, ar_quran, de_quran) in enumerate(quran_matches, start=1):
            blocks.append(f"""
Candidate {idx} (Score={score:.3f}):
Arabic: {ar_quran}
German: {de_quran}
            """.strip())

        quran_hint = """
HINT - POSSIBLE QURAN VERSES:

The following are candidate Quran verses that may appear in the text.
Rules:
- Use the German translation ONLY if the verse clearly appears in the Arabic text
- Do NOT add verses that don't appear in the current text
- Do NOT modify the provided translations

Candidates:
        """.strip() + "\n\n" + "\n\n".join(blocks)

    # 3) GPT translation
    system_prompt = (
        "You are a professional Arabic-German translator for religious texts, sermons, and classical Arabic. "
        "Translate accurately and completely. Preserve religious terminology (Allah, Umma, Sunnah, etc.). "
        "Output ONLY the translation, no comments or formatting."
    )

    user_prompt = f"""
Translate the following Arabic text to German.
- Translate ONLY the Arabic text below
- Do NOT add content not in the original
- Preserve religious terms accurately

{quran_hint}

Arabic text:
{txt}

German translation:
    """.strip()

    try:
        resp = client.chat.completions.create(
            model=TRANSLATION_MODEL,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ]
        )
        return resp.choices[0].message.content.strip()
    except Exception as e:
        print(f"Translation error: {e}")
        return ""

In [ ]:
# -------------------------------------------------------------------
# TEST FUNCTIONS
# -------------------------------------------------------------------

def test_rag(text: str, min_similarity: float = RAG_MIN_SIMILARITY, top_k: int = RAG_TOP_K, translate: bool = True):
    """
    Test RAG matching on a single Arabic text.
    Shows all Quran matches and optionally the full translation.
    """
    txt = (text or "").strip()
    if not txt:
        print("No text provided.")
        return

    print("=" * 80)
    print("Arabic input:")
    print(txt)
    print("-" * 80)

    # RAG matching
    matches = match_quran_rag(txt, min_similarity=min_similarity, top_k=top_k)

    if not matches:
        print(f"No Quran matches with score ≥ {min_similarity:.2f}")
    else:
        print(f"{len(matches)} Quran match(es) with score ≥ {min_similarity:.2f}:\n")
        for idx, (score, ar_quran, de_quran) in enumerate(matches, start=1):
            print(f"[Match {idx}] Score: {score:.3f}")
            print(f"Arabic: {ar_quran}")
            print(f"German: {de_quran}")
            print("-" * 40)

    # Full translation
    if translate:
        print("\nFull translation:")
        translation = translate_text(txt)
        print(translation)

    print("=" * 80 + "\n")


def test_rag_batch(texts: list[str], **kwargs):
    """Test RAG on a list of Arabic texts."""
    for i, t in enumerate(texts, start=1):
        print(f"\n### Test {i} ###")
        test_rag(t, **kwargs)

## Test Examples

Run the cells below to test the RAG system with sample verses.

In [ ]:
# Test with Al-Fatiha opening
test_rag("الحمد لله رب العالمين")

In [ ]:
# Test with multiple verses
test_verses = [
    "الحمد لله رب العالمين",
    "ولا تحسبن الله غافلا عما يعمل الظالمون",
    "فبأي الاء ربكما تكذبان",
    "يا أيها الذين أمنوا اتقوا الله حق تقاته ولا تموتن إلا وأنتم مسلمون",
]

test_rag_batch(test_verses, translate=True)

In [ ]:
# Test with mixed text (sermon with Quran quotes)
sermon_text = """
أما بعد فأوصيكم ونفسي بتقوى الله كما قال تعالى في كتابه الكريم: 
يا أيها الذين أمنوا اتقوا الله حق تقاته ولا تموتن إلا وأنتم مسلمون
"""

test_rag(sermon_text)

In [ ]:
# Test Athan detection
athan_phrases = [
    "الله أكبر الله أكبر",
    "أشهد أن لا إله إلا الله",
    "حي على الصلاة",
]

print("Testing Athan detection:")
print("=" * 50)
for phrase in athan_phrases:
    score, de, ar = fuzzy_match_athan(phrase)
    print(f"Input: {phrase}")
    print(f"Match: {ar} (score={score:.2f})")
    print(f"Translation: {de}")
    print("-" * 50)

In [ ]:
# Custom test - enter your own text
your_text = """بسم الله الرحمن الرحيم"""

test_rag(your_text, min_similarity=0.5, top_k=10)